# Lakehouse Maintenance - DROP TABLES (destructive)

Bulk-drops Delta tables in a schema-enabled Lakehouse whose names match a
SQL `LIKE` pattern. Use for tearing down staging output, retiring a source
prefix, or cleaning up after a failed load.

> **⚠️ Destructive - not schedule-safe.** Unlike the other notebooks in this
> folder, this one *removes* tables. Run it interactively, review the dry-run
> output, then flip `DRY_RUN` off. Do not put it on a timer.

| Guardrail | Behavior |
|-----------|----------|
| **`DRY_RUN`** | Defaults to `True`. Lists what *would* be dropped and touches nothing. Set to `False` for a real run. |
| **`ALLOW_FULL_SCHEMA`** | Defaults to `False`. A pattern that matches every table (e.g. `%`) is refused unless you opt in explicitly. |

**Schema-aware:** Resolves tables via `SHOW TABLES IN <schema>` against the attached Lakehouse.

## Configuration

Target workspace and Lakehouse are auto-detected from the attached Lakehouse - override the constants below to point at a different environment. Each entry in `DROP_TARGETS` pairs a schema with a SQL `LIKE` pattern (`%` = any run of characters, `_` = any single character).

In [ ]:
# ── CONFIGURATION ────────────────────────────────────────────────────────

context = notebookutils.runtime.context

# --- Lakehouse Target ---
# Auto-detected from the notebook's attached lakehouse.
# Override these if targeting a different workspace or lakehouse.
WORKSPACE_NAME = context.get("currentWorkspaceName")
LAKEHOUSE_NAME = context.get("defaultLakehouseName")

# --- Drop Targets ---
# One entry per (schema, LIKE pattern) to sweep. Patterns are SQL LIKE:
#   %  matches any run of characters (including none)
#   _  matches exactly one character
# Examples (replace with your own - these are illustrative only):
DROP_TARGETS = [
    {"schema": "<SchemaName>", "pattern": "<Prefix>%"},
    # {"schema": "bronze", "pattern": "stg_%"},
    # {"schema": "silver", "pattern": "tmp_load_2025_%"},
]

# --- Dry Run ---
# True  (default) - list tables that WOULD be dropped; change nothing.
# False           - actually DROP the matched tables. Review a dry run first.
DRY_RUN = True

# --- Full-Schema Guard ---
# A pattern like "%" matches every table in the schema. That is almost never
# what you want by accident, so it is refused unless you opt in here.
# False (default) - abort if any target would match every table in its schema.
# True            - permit whole-schema wipes. Use with extreme care.
ALLOW_FULL_SCHEMA = False

print("--- Configuration ---")
print(f"  Workspace:         {WORKSPACE_NAME}")
print(f"  Lakehouse:         {LAKEHOUSE_NAME}")
print(f"  Drop targets:      {len(DROP_TARGETS)}")
print(f"  Dry run:           {DRY_RUN}")
print(f"  Allow full schema: {ALLOW_FULL_SCHEMA}")
if DRY_RUN:
    print("\n  DRY_RUN is ON - no tables will be dropped.")
else:
    print("\n  *** DRY_RUN is OFF - matched tables WILL be dropped. ***")

## Resolve Matches

For each target, enumerate the schema's tables and keep the ones matching the
`LIKE` pattern. Nothing is dropped in this cell - it only builds the work list.
A pattern that would match every table in its schema is blocked here unless
`ALLOW_FULL_SCHEMA` is set.

In [ ]:
import re


def like_to_regex(pattern: str) -> str:
    """Translate a SQL LIKE pattern into an anchored regex.

    Regex metacharacters in the pattern are escaped first so only the LIKE
    wildcards carry special meaning: ``%`` -> ``.*`` and ``_`` -> ``.``.

    Args:
        pattern: A SQL LIKE pattern, e.g. ``"stg_%"``.

    Returns:
        An anchored regex string suitable for ``re.fullmatch``.
    """
    escaped = re.escape(pattern)
    escaped = escaped.replace("%", ".*").replace("_", ".")
    return f"^{escaped}$"


def matches_all(pattern: str) -> bool:
    """Return True if the pattern matches every possible table name.

    True for an empty pattern or one built only from ``%`` wildcards
    (e.g. ``""``, ``"%"``, ``"%%"``).

    Args:
        pattern: A SQL LIKE pattern.

    Returns:
        Whether the pattern is unconstrained.
    """
    return pattern == "" or set(pattern) <= {"%"}


def resolve_target(schema: str, pattern: str) -> list[str]:
    """List existing tables in ``schema`` whose names match ``pattern``.

    Args:
        schema: Schema (namespace) to scan in the attached Lakehouse.
        pattern: SQL LIKE pattern applied to table names.

    Returns:
        Backtick-quoted ``schema.table`` identifiers, sorted by name.
    """
    regex = like_to_regex(pattern)
    rows = spark.sql(f"SHOW TABLES IN `{schema}`").collect()
    matched = [
        f"`{schema}`.`{r['tableName']}`"
        for r in rows
        if r["tableName"] and re.fullmatch(regex, r["tableName"])
    ]
    return sorted(matched)


# Guard: block unconstrained patterns unless explicitly allowed.
full_schema_hits = [
    t for t in DROP_TARGETS if matches_all(t["pattern"]) and not ALLOW_FULL_SCHEMA
]
if full_schema_hits:
    offending = ", ".join(f"{t['schema']}:'{t['pattern']}'" for t in full_schema_hits)
    raise ValueError(
        f"Pattern(s) match every table in their schema ({offending}). "
        f"Set ALLOW_FULL_SCHEMA = True to permit a whole-schema wipe."
    )

# Build the work list.
to_drop = []  # list[dict]: {"schema", "pattern", "table"}
for target in DROP_TARGETS:
    schema = target["schema"]
    pattern = target["pattern"]
    try:
        tables = resolve_target(schema, pattern)
    except Exception as exc:
        print(f"  WARN: could not scan schema '{schema}': {exc}")
        continue

    print(f"  {schema} LIKE '{pattern}' -> {len(tables)} match(es)")
    for t in tables:
        to_drop.append({"schema": schema, "pattern": pattern, "table": t})

print(f"\nResolved {len(to_drop)} table(s) to drop.")

## Drop (or Preview)

With `DRY_RUN = True` this only prints the resolved tables. With `DRY_RUN = False`
it issues `DROP TABLE IF EXISTS` for each match and records the outcome.

In [ ]:
results = {
    "dropped": [],  # list[str]: "`schema`.`table`"
    "failed":  [],  # list[dict]: {"table": str, "error": str}
}

if DRY_RUN:
    print("DRY RUN - the following tables WOULD be dropped:\n")
    for item in to_drop:
        print(f"  {item['table']}")
    print(f"\nTotal: {len(to_drop)} table(s). Set DRY_RUN = False to drop them.")
else:
    print(f"Dropping {len(to_drop)} table(s)...\n")
    for item in to_drop:
        table = item["table"]
        try:
            print(f"  DROP {table} ...", end=" ")
            spark.sql(f"DROP TABLE IF EXISTS {table}")
            print("OK")
            results["dropped"].append(table)
        except Exception as exc:
            print(f"FAILED: {exc}")
            results["failed"].append({"table": table, "error": str(exc)})

## Summary

In [ ]:
print("\n" + "─" * 80)
print("  Summary")
print("─" * 80)
print(f"  Mode:      {'DRY RUN (nothing dropped)' if DRY_RUN else 'LIVE'}")
print(f"  Matched:   {len(to_drop)}")

if not DRY_RUN:
    print(f"  Dropped:   {len(results['dropped'])}")
    print(f"  Failed:    {len(results['failed'])}")
    for entry in results["failed"]:
        print(f"    - {entry['table']}: {entry['error']}")

print("─" * 80)

if not DRY_RUN and results["failed"]:
    raise RuntimeError(
        f"{len(results['failed'])} table(s) failed to drop. See summary above."
    )